## 1. Setup & Installation\n

In [9]:
# ============================================================
# Cell 1: Mount Drive & Unzip Data
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

!cp "/content/drive/MyDrive/movies1000.zip" "/content/"
!unzip -o -q /content/movies1000.zip -d /content/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
# ============================================================
# Cell 2: Imports & Seed
# ============================================================
!pip install -q transformers datasets accelerate scikit-learn pandas sentencepiece

import os, random, warnings
import numpy as np
import torch
import pandas as pd
from pathlib import Path
from datasets import Dataset, ClassLabel
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 2. Data Loading\n

In [11]:
# ============================================================
# Cell 3: Data Loading (FIXED — ClassLabel cast + no typo)
# ============================================================
def load_data(data_dir):
    reviews, labels = [], []
    pos_dir = Path(data_dir) / 'pos'
    neg_dir = Path(data_dir) / 'neg'

    for fp in sorted(pos_dir.glob('*.txt')):
        reviews.append(fp.read_text(encoding='utf-8').strip())
        labels.append(1)
    for fp in sorted(neg_dir.glob('*.txt')):
        reviews.append(fp.read_text(encoding='utf-8').strip())
        labels.append(0)

    return pd.DataFrame({'text': reviews, 'label': labels})

data_dir = "/content/movies1000"
df = load_data(data_dir)
print(f"Total samples: {len(df)}")
print(f"Label distribution:\n{df['label'].value_counts()}")

# Convert to HuggingFace Dataset and cast label to ClassLabel
dataset = Dataset.from_pandas(df)
dataset = dataset.cast_column("label", ClassLabel(names=["neg", "pos"]))

# Now stratify works
split = dataset.train_test_split(test_size=0.2, seed=SEED, stratify_by_column="label")
train_dataset = split['train']
eval_dataset = split['test']
print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

Total samples: 2000
Label distribution:
label
1    1000
0    1000
Name: count, dtype: int64


Casting the dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Train: 1600, Eval: 400


In [12]:
# ============================================================
# Cell 4: Tokenization (RoBERTa + Head/Tail Truncation)
# ============================================================
from transformers import AutoTokenizer, DataCollatorWithPadding

model_name = "roberta-base" # The heavy lifter, no gamma/beta bugs
tokenizer = AutoTokenizer.from_pretrained(model_name)
MAX_LEN = 512

def truncate_head_tail(text, head_words=128, tail_words=384):
    """Keeps the beginning and end of a long review, cutting the middle."""
    words = text.split()
    if len(words) <= (head_words + tail_words):
        return text
    return " ".join(words[:head_words] + words[-tail_words:])

def tokenize(examples):
    # Apply the truncation logic before tokenizing
    truncated_texts = [truncate_head_tail(doc) for doc in examples["text"]]
    return tokenizer(truncated_texts, truncation=True, max_length=MAX_LEN, padding=False)

encoded_train = train_dataset.map(tokenize, batched=True, remove_columns=["text"])
encoded_eval = eval_dataset.map(tokenize, batched=True, remove_columns=["text"])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(f"Encoded — train: {len(encoded_train)}, eval: {len(encoded_eval)}")

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Encoded — train: 1600, eval: 400


## 3. Preprocessing (Head+Tail Truncation)\n

## 4. Model Prep & LLRD Configuration\n

In [13]:
# ============================================================
# Cell 5 & 6: RoBERTa Training + LLRD + Early Stopping
# ============================================================
from transformers import AutoConfig, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef
import torch
import numpy as np

# 1. LOAD MODEL
config = AutoConfig.from_pretrained(
    model_name,
    num_labels=2,
    hidden_dropout_prob=0.2,
    attention_probs_dropout_prob=0.2
)
model = AutoModelForSequenceClassification.from_pretrained(model_name, config=config)

# 2. LAYER-WISE LEARNING RATE DECAY (LLRD) OPTIMIZER
def create_llrd_optimizer(model, base_lr=1e-5, weight_decay=0.15, layer_decay=0.95):
    opt_parameters = []

    # The classification head gets the standard base_lr
    opt_parameters.append({"params": model.classifier.parameters(), "lr": base_lr, "weight_decay": weight_decay})

    # The RoBERTa layers get progressively smaller learning rates
    current_lr = base_lr
    for layer in reversed(model.roberta.encoder.layer):
        opt_parameters.append({"params": layer.parameters(), "lr": current_lr, "weight_decay": weight_decay})
        current_lr *= layer_decay

    # The absolute lowest learning rate goes to the fundamental embeddings
    opt_parameters.append({"params": model.roberta.embeddings.parameters(), "lr": current_lr, "weight_decay": weight_decay})

    return torch.optim.AdamW(opt_parameters)

custom_optimizer = create_llrd_optimizer(model)

# 3. CONFIGURE TRAINING
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=6,                 # Increased epochs, relying on Early Stopping
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    save_total_limit=2,
    dataloader_num_workers=2,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="binary"),
        "mcc": matthews_corrcoef(labels, preds),
    }

# 4. TRAIN WITH CALLBACKS
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_train,
    eval_dataset=encoded_eval,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(custom_optimizer, None), # Injecting our custom LLRD optimizer
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] # Stops if F1 doesn't improve for 2 epochs
)

print(f"Starting optimized training with {model_name}...")
trainer.train()

print("\nFinal Evaluation:")
results = trainer.evaluate()
print(f"  Accuracy: {results['eval_accuracy']:.4f}")
print(f"  F1:       {results['eval_f1']:.4f}")
print(f"  MCC:      {results['eval_mcc']:.4f}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting optimized training with roberta-base...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Mcc
1,0.442029,0.322928,0.885000,0.881443,0.771390
2,0.290240,0.225000,0.920000,0.915789,0.844232
3,0.124192,0.198586,0.935000,0.935644,0.870174
4,0.220698,0.226353,0.932500,0.934307,0.866311
5,0.122374,0.265945,0.930000,0.932039,0.861552


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Final Evaluation:


  Accuracy: 0.9350
  F1:       0.9356
  MCC:      0.8702


In [14]:
# ============================================================
# MASTER SCRIPT: Phase 2 (Pseudo-Labeling), Phase 3 (Student), Phase 4 (CSV)
# ============================================================
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from datasets import load_dataset, concatenate_datasets
from transformers import AutoConfig, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback

# ------------------------------------------------------------
# STEP A: Mounting Drive & Loading the TXT file
# ------------------------------------------------------------
from google.colab import drive
from datasets import load_dataset # Ensure this is imported

print("Mounting Google Drive...")
drive.mount('/content/drive')

print("Loading the raw test text file from Drive...")
# IMPORTANT: If testSentiment.txt is inside a specific folder, update this path.
# Example: "/content/drive/MyDrive/Colab Notebooks/testSentiment.txt"
drive_file_path = "/content/drive/MyDrive/testSentiment.txt"

raw_test_dataset = load_dataset("text", data_files=drive_file_path)
test_dataset = raw_test_dataset["train"]

print("Encoding the unlabelled test set...")
# Since it's a txt file, HF automatically names the column "text", so we map and remove it
encoded_test = test_dataset.map(tokenize, batched=True, remove_columns=["text"])
# ------------------------------------------------------------
# STEP B: Phase 2 - Pseudo-Labeling with the Teacher (FIXED)
# ------------------------------------------------------------
print("\nRunning inference with Teacher model to find confident labels...")
test_predictions = trainer.predict(encoded_test) # 'trainer' is your Phase 1 model
logits = torch.tensor(test_predictions.predictions)

# Convert to probabilities
probs = F.softmax(logits, dim=-1)
max_probs, predicted_labels = torch.max(probs, dim=-1)

# TWEAK: Lowered to 95% to extract a meaningful amount of data
CONFIDENCE_THRESHOLD = 0.9
confident_indices = torch.where(max_probs > CONFIDENCE_THRESHOLD)[0].tolist()

print(f"Total test samples: {len(encoded_test)}")
print(f"Highly confident samples extracted: {len(confident_indices)}")

# Extract the confident data
pseudo_labeled_dataset = encoded_test.select(confident_indices)
confident_labels = predicted_labels[confident_indices].tolist()
pseudo_labeled_dataset = pseudo_labeled_dataset.add_column("label", confident_labels)

columns_to_keep = ["input_ids", "attention_mask", "label"]
pseudo_labeled_clean = pseudo_labeled_dataset.select_columns(columns_to_keep)
encoded_train_clean = encoded_train.select_columns(columns_to_keep)

# THE BUG FIX: Force the new dataset features to perfectly match the old dataset features
pseudo_labeled_clean = pseudo_labeled_clean.cast(encoded_train_clean.features)

# Merge and shuffle
combined_train_dataset = concatenate_datasets([encoded_train_clean, pseudo_labeled_clean])
combined_train_dataset = combined_train_dataset.shuffle(seed=42)
print(f"New Student Training Set Size: {len(combined_train_dataset)} samples!\n")

# ------------------------------------------------------------
# STEP C: Phase 3 - Training the Student Model
# ------------------------------------------------------------
print("Initializing a fresh Student model with relaxed hyperparameters...")

# Revert to standard Dropout for the larger dataset
student_config = AutoConfig.from_pretrained(
    model_name,
    num_labels=2,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1
)
student_model = AutoModelForSequenceClassification.from_pretrained(model_name, config=student_config)

# Revert Weight Decay and Learning Rate using your LLRD function
student_optimizer = create_llrd_optimizer(
    student_model,
    base_lr=2e-5,       # Bumped back up for the larger dataset
    weight_decay=0.05   # Lowered back to standard
)

student_training_args = TrainingArguments(
    output_dir="./student_results",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    save_total_limit=1,
)

student_trainer = Trainer(
    model=student_model,
    args=student_training_args,
    train_dataset=combined_train_dataset,
    eval_dataset=encoded_eval,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(student_optimizer, None),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

print("Starting final Student training...")
student_trainer.train()

print("\nFinal Student Evaluation:")
student_results = student_trainer.evaluate()
print(f"  Accuracy: {student_results['eval_accuracy']:.4f}")
print(f"  F1:       {student_results['eval_f1']:.4f}")


Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading the raw test text file from Drive...
Encoding the unlabelled test set...

Running inference with Teacher model to find confident labels...


Total test samples: 25000
Highly confident samples extracted: 760


Flattening the indices:   0%|          | 0/760 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/760 [00:00<?, ? examples/s]

New Student Training Set Size: 2360 samples!

Initializing a fresh Student model with relaxed hyperparameters...


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Final Student Evaluation:


  Accuracy: 0.9475
  F1:       0.9481


In [15]:

# ------------------------------------------------------------
# STEP D: Phase 4 - Final Auto-Grader CSV Generation
# ------------------------------------------------------------
print("\nGenerating final submission file...")
final_predictions = student_trainer.predict(encoded_test)
final_class_indices = np.argmax(final_predictions.predictions, axis=-1)

label_map = {1: 'P', 0: 'N'}
final_labels = [label_map[idx] for idx in final_class_indices]

# Dump the raw list directly into a DataFrame (no column names)
submission_df = pd.DataFrame(final_labels)

submission_file = "/submission-movie-3.csv"

# index=False drops the row numbers, header=False drops the column titles
submission_df.to_csv(submission_file, index=False, header=False)

print(f" Success! Barebones file ready for the auto-grader: {submission_file}")


Generating final submission file...


 Success! Barebones file ready for the auto-grader: /submission-movie-3.csv


In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
from google.colab import files
files.download("/submission-movie-3.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>